# ML-04 — Search Intelligence Data Contract

This notebook turns the Search Intelligence / content-refresh lane into a small, auditable ML data contract using the FlyRank internship warehouse.

**Development month:** March 2026  
**Decision moment:** end of 2026-03-15  
**Feature window:** 2026-03-01 through 2026-03-15  
**Outcome window:** 2026-03-16 through 2026-03-31

The later final month is not used for feature or label design here.

## Setup

The Hugging Face token is read from a Colab Secret named `HF_TOKEN`; the token itself is never written into the notebook.

The notebook downloads only the March daily-performance partition plus `dim_content`, then uses DuckDB locally to query the Parquet files.

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download
import duckdb
import pandas as pd
import numpy as np

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise ValueError("HF_TOKEN not found in Colab Secrets.")

api = HfApi()
me = api.whoami(token=hf_token)
print("Token account:", me["name"])

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=hf_token,
)

content_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_content.parquet",
    token=hf_token,
)

con = duckdb.connect()

print("March fact data ready.")
print("Content dimension ready.")

## 1. Unit of analysis + time window

**One row in the raw March fact table:** one daily observation for a pseudonymized content item belonging to a pseudonymized client.

The raw grain is therefore:

`report_date × client_hash_id × content_hash_id`

**Tables used:**  
- `fact_content_daily_performance` for daily Google Search Console performance and availability.  
- `dim_content` for content metadata, specifically `content_created_date`.

**Development time window:** March 2026. I use March 1–15 as the information available at the decision moment and March 16–31 as the later outcome window.

**Prediction / ranking goal:** rank content items for human review using an observed search-visibility decline proxy. The proxy supports a possible refresh decision; it does **not** claim that a refresh will cause performance to improve.

**Deliberately excluded:** future-window measurements and anything derived from them are excluded from the honest feature set because they would reveal the outcome. Pseudonymous IDs are kept only for grouping/joining, not as model features.

## 2. Fields: feature / label / context / excluded

### Features — five maximum

1. **`past_impressions_per_day`** — knowable at the decision moment because it uses only valid GSC observations from March 1–15.
2. **`past_clicks_per_day`** — knowable at the decision moment because it uses only valid GSC observations from March 1–15.
3. **`past_avg_position`** — knowable at the decision moment because it is aggregated only from March 1–15 Google Search position observations.
4. **`past_ctr`** — knowable at the decision moment because it is computed only from March 1–15 clicks and impressions.
5. **`content_age_days`** — knowable at the decision moment because it uses the content creation date and the fixed decision date, March 15.

### Label / proxy

**`is_declining`** is a directional proxy for a potential refresh opportunity.

A content item is eligible for the proxy when:
- it has at least **8 valid GSC days** in March 1–15,
- at least **8 valid GSC days** in March 16–31, and
- it had positive past impressions.

For eligible content, `is_declining = 1` when average daily impressions in March 16–31 are **at least 20% lower** than average daily impressions in March 1–15.

This 20% cutoff is a simple heuristic to avoid labeling tiny day-to-day changes as meaningful decline.

### Context

`client_hash_id`, `content_hash_id`, `report_date`, `month`, and `gsc_data_available` are used for identity, grouping, filtering, joins, or time logic. They are not learned as model features.

### Excluded

- Future impressions and any future-derived ratios: excluded because they directly reveal the proxy label.
- `is_declining`: excluded from `X` because it is the target.
- Pseudonymous IDs: excluded from `X` because they are identifiers rather than generalizable predictive signals.
- GA4 fields: not used in this first contract so that mixed GA4 historical coverage does not complicate the GSC-focused baseline.

## 3. Verify it with queries (grain, counts, missing values, windows)

The three cells below are the notebook's **three verification queries**. Each one checks a specific contract claim.

### Verification query 1 — raw grain + row count

If `report_date + client_hash_id + content_hash_id` is truly the raw grain, the total row count should equal the number of distinct combinations.

In [ ]:
grain_check = con.sql(
    f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(
            DISTINCT (
                report_date,
                client_hash_id,
                content_hash_id
            )
        ) AS unique_grain_rows
    FROM read_parquet('{march_path}')
    """
).df()

grain_check

Expected from the March slice: **9,841,378 total rows and 9,841,378 unique grain rows**, supporting the stated raw grain.

### Verification query 2 — date span / window

This verifies the actual dates inside the file rather than trusting the folder name.

In [ ]:
window_check = con.sql(
    f"""
    SELECT
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT report_date) AS distinct_dates,
        COUNT(DISTINCT month) AS distinct_months,
        MIN(month) AS month_value
    FROM read_parquet('{march_path}')
    """
).df()

window_check

Expected from the March slice: **2026-03-01 through 2026-03-31**, 31 distinct dates, and one month value (`2026-03`).

### Verification query 3 — GSC availability + missingness

A measured zero and unavailable data are not the same thing. This query keeps only rows where GSC data is explicitly available and checks missingness in the core GSC fields.

In [ ]:
availability_check = con.sql(
    f"""
    SELECT
        COUNT(*) AS rows_after_gsc_filter,
        COUNT(DISTINCT client_hash_id) AS clients_after_gsc_filter,
        COUNT(DISTINCT content_hash_id) AS content_after_gsc_filter,

        SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END)
            AS missing_impressions,

        SUM(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END)
            AS missing_clicks,

        SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END)
            AS missing_avg_position

    FROM read_parquet('{march_path}')
    WHERE gsc_data_available IS TRUE
    """
).df()

availability_check

The earlier availability check showed **3,611,061** GSC-available daily observations, covering **47 clients** and **176,738 content items**. The executed output above also shows whether the core GSC measurements are missing inside that available slice.

### Build the five-feature frame and proxy label

The raw warehouse grain is daily. The modeling grain below becomes **one row per client-content item at the March 15 decision point**.

Only March 1–15 contributes honest features. March 16–31 is used only to construct the later proxy outcome.

In [ ]:
analysis_df = con.sql(
    f"""
    WITH page_windows AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                     AND gsc_data_available IS TRUE
                    THEN gsc_impressions ELSE 0
                END
            ) AS past_impressions,

            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                     AND gsc_data_available IS TRUE
                    THEN gsc_clicks ELSE 0
                END
            ) AS past_clicks,

            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                     AND gsc_data_available IS TRUE
                    THEN gsc_sum_position ELSE 0
                END
            ) AS past_sum_position,

            COUNT(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                     AND gsc_data_available IS TRUE
                    THEN 1
                END
            ) AS past_available_days,

            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
                     AND gsc_data_available IS TRUE
                    THEN gsc_impressions ELSE 0
                END
            ) AS future_impressions,

            COUNT(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
                     AND gsc_data_available IS TRUE
                    THEN 1
                END
            ) AS future_available_days

        FROM read_parquet('{march_path}')
        GROUP BY client_hash_id, content_hash_id
    )

    SELECT
        w.client_hash_id,
        w.content_hash_id,

        w.past_impressions::DOUBLE / w.past_available_days
            AS past_impressions_per_day,

        w.past_clicks::DOUBLE / w.past_available_days
            AS past_clicks_per_day,

        w.past_sum_position::DOUBLE / NULLIF(w.past_impressions, 0)
            AS past_avg_position,

        w.past_clicks::DOUBLE / NULLIF(w.past_impressions, 0)
            AS past_ctr,

        DATE_DIFF(
            'day',
            c.content_created_date,
            DATE '2026-03-15'
        ) AS content_age_days,

        w.past_available_days,

        w.future_impressions::DOUBLE / w.future_available_days
            AS future_impressions_per_day,

        CASE
            WHEN
                (w.future_impressions::DOUBLE / w.future_available_days)
                <= 0.80 * (w.past_impressions::DOUBLE / w.past_available_days)
            THEN 1
            ELSE 0
        END AS is_declining

    FROM page_windows AS w
    INNER JOIN read_parquet('{content_path}') AS c
        ON w.client_hash_id = c.client_hash_id
       AND w.content_hash_id = c.content_hash_id

    WHERE
        w.past_available_days >= 8
        AND w.future_available_days >= 8
        AND w.past_impressions > 0
        AND c.content_created_date IS NOT NULL
        AND c.content_created_date <= DATE '2026-03-15'
    """
).df()

FEATURE_COLS = [
    "past_impressions_per_day",
    "past_clicks_per_day",
    "past_avg_position",
    "past_ctr",
    "content_age_days",
]

model_df = analysis_df[
    ["client_hash_id", "content_hash_id"] + FEATURE_COLS + ["is_declining"]
].dropna().copy()

print("Model rows:", len(model_df))
print(
    "Duplicate client-content rows:",
    model_df.duplicated(["client_hash_id", "content_hash_id"]).sum()
)
print("\nMissing values:")
print(model_df[FEATURE_COLS + ["is_declining"]].isna().sum())
print("\nDeclining rate:", round(model_df["is_declining"].mean(), 3))

display(model_df.head())

### Deliberate leakage experiment

This is intentionally wrong once, to demonstrate why leakage is dangerous.

The honest model uses only the five fields available by March 15. Then one **future-derived** field, `future_visibility_ratio`, is added. Because the label itself is defined from the future/past visibility ratio, this leaked field contains the answer in disguised form.

The score here is only a quick leakage demonstration, not the final lane success metric.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X = model_df[FEATURE_COLS]
y = model_df["is_declining"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

honest_model = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=100,
    random_state=42,
)
honest_model.fit(X_train, y_train)
honest_accuracy = accuracy_score(y_test, honest_model.predict(X_test))

# Intentionally leaked field: it uses the future outcome window.
leak_df = model_df[
    ["client_hash_id", "content_hash_id"] + FEATURE_COLS + ["is_declining"]
].merge(
    analysis_df[
        ["client_hash_id", "content_hash_id", "future_impressions_per_day"]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="left",
    validate="one_to_one",
)

leak_df["future_visibility_ratio"] = (
    leak_df["future_impressions_per_day"]
    / leak_df["past_impressions_per_day"]
)

LEAKED_FEATURES = FEATURE_COLS + ["future_visibility_ratio"]

X_leak = leak_df[LEAKED_FEATURES].replace([np.inf, -np.inf], np.nan)
valid = X_leak.notna().all(axis=1)

X_leak = X_leak.loc[valid]
y_leak = leak_df.loc[valid, "is_declining"]

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leak,
    y_leak,
    test_size=0.25,
    random_state=42,
    stratify=y_leak,
)

leaked_model = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=100,
    random_state=42,
)
leaked_model.fit(Xl_train, yl_train)
leaked_accuracy = accuracy_score(yl_test, leaked_model.predict(Xl_test))

print(f"Honest quick accuracy: {honest_accuracy:.3f}")
print(f"Leaked quick accuracy: {leaked_accuracy:.3f}")
print(f"Score jump: {leaked_accuracy - honest_accuracy:+.3f}")

The leaked score is not a legitimate improvement. `future_visibility_ratio` uses March 16–31 information and directly exposes the rule used to construct `is_declining`.

The honest feature frame therefore **removes the leaked field and keeps only the original five features**.

In [ ]:
# Remove the deliberately leaked feature and keep the honest frame.
if "future_visibility_ratio" in leak_df.columns:
    leak_df = leak_df.drop(columns=["future_visibility_ratio"])

final_feature_frame = model_df.copy()

assert "future_visibility_ratio" not in final_feature_frame.columns
assert set(FEATURE_COLS).issubset(final_feature_frame.columns)

print("Leak removed.")
print("Final honest feature columns:", FEATURE_COLS)
print(f"Honest quick accuracy retained: {honest_accuracy:.3f}")

## 4. Data limits

This slice cannot establish that a content refresh **causes** search performance to improve.

Important limitations:

- **Incomplete GSC coverage:** only the GSC-observed subset is used for search-performance measurements, so the analysis does not represent every March warehouse row.
- **Short-window proxy:** `is_declining` compares two short halves of one month. Search demand, ranking volatility, events, or seasonality can move impressions even when the content itself did not materially change.
- **Heuristic threshold:** the 20% decline cutoff and 8-day minimum-coverage rule are practical choices for this exercise, not universal truths.
- **Decision support, not causal truth:** observed visibility decline is a proxy for “worth reviewing,” not proof that the page needs a refresh or that refreshing it would improve results.
- **GSC-focused first version:** GA4 features are deliberately left out here rather than pretending mixed analytics-history coverage is fully comparable.

The output should therefore be interpreted as **directional decision support for human review**.

## Self-check

Before submission:

- [x] Contract states the raw grain, tables, time window, proxy, and deliberate exclusion.
- [x] Exactly three verification queries are clearly labeled.
- [x] Availability is checked with `IS TRUE`.
- [x] The feature frame contains five honest features maximum.
- [x] Every feature has an “available at the decision moment” explanation.
- [x] A label-derived leaked field is demonstrated and then removed.
- [x] One named limitation of the slice is stated.
- [x] Claims use careful language: observed, measured, directional, decision-support.
- [ ] Run **Runtime → Run all** successfully in Colab.
- [ ] Commit this notebook under `work/notebooks/w03_data_contract.ipynb`.
- [ ] Submit the repository URL on the assignment card.

**Do not commit any Hugging Face token or private client/query data.**